
Currently intended usage pattern for `llm_function`:

- define typed tools
- mark them with `llm_function_tools`
- pass them into `llm_function` through a tool source
- bundle runtime settings and tools into config
- call the decorated function like a normal Python function


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

from pydantic import BaseModel, Field
from typing import Type, List, Optional

from llm_function_tools import llm_tool
from llm_function import InMemoryToolSource, PythonFileToolSource, LlmFunctionConfig, LlmFunctionRuntime, LlmRuntimeConfig, llm_function


## Define available tools

The runtime can assemble workflows only from tools you expose through tool sources. Tools can be defined directly in the current notebook or kept in a separate `.py` file.


In [2]:
class GetWeatherInput(BaseModel):
    city: str = Field(..., description="City name.")


class GetWeatherOutput(BaseModel):
    forecast: str = Field(..., description="Weather forecast for the city.")


@llm_tool(tags=["weather"])
def get_weather(inputs: GetWeatherInput) -> GetWeatherOutput:
    """Get current weather for a city."""
    return GetWeatherOutput(forecast=f"Sunny in {inputs.city}")


class EmailInformationPoint(BaseModel):
    title: str = Field(None, description="Few word description of the information.")
    content: str = Field(..., description="Content of the information.")

class SendReportEmailInput(BaseModel):
    city: str = Field(..., description="Name of the city where report will be send.")
    information: list[EmailInformationPoint]

class SendReportEmailOutput(BaseModel):
    email_sent: bool = Field(..., description="Conformation that email was send successfully.")
    message: str = Field(None, description="Optional comments from the process.")

@llm_tool(tags=["email"])
def send_report_email(inputs: SendReportEmailInput) -> SendReportEmailOutput:
    """Sends a report email with given information points to a city."""

    if inputs.city not in ["London", "Berlin"]:
        return SendReportEmailOutput(
            email_sent = False,
            message = f"Email was not sent to {inputs.city}!"
        )
    else:
        return SendReportEmailOutput(
            email_sent = True,
            message = f"Email sent to {inputs.city}!"
        )

class QueryDatabaseInput(BaseModel):
    topic: str = Field(..., description="Topic of a requested piece of information.")
    location: str = Field(..., description="Filter for location name.")
    uid: str = Field(None, description="Filter for unique indentifier of the database item.")

class QueryDatabaseOutput(BaseModel):
    info: str = Field(..., description="Content of the information.")
    uid: str = Field(None, description="Unique indentifier of the database item.")

@llm_tool(tags=["database"])
def query_database(inputs : QueryDatabaseInput) -> QueryDatabaseOutput:
    """Get information from the database with provided filters."""
    return QueryDatabaseOutput(
        info = f"Content extracted from the database for {inputs.topic} in {inputs.location} is ...",
        uid = "0000"
    )

class QueryWebInput(BaseModel):
    search_input: str = Field(..., description="Topic to be searched on the web.")


class QueryWebOutput(BaseModel):
    search_results: List[str] = Field(..., description="List relevant info from search results.")

@llm_tool(tags=["web"])
def query_web(inputs : QueryWebInput) -> QueryWebOutput:
    """Get information from the internet for provided query."""
    return QueryWebOutput(
        search_results = ["Relevant content found in first search result is ..."],
    )


tool_sources = [InMemoryToolSource([get_weather, send_report_email, query_database, query_web])]


You can also keep tools in a separate `.py` file and load them with `PythonFileToolSource`:


In [3]:
tmp_dir = TemporaryDirectory()
tool_file = Path(tmp_dir.name) / "weather_tools.py"

tool_file.write_text(
    """
from pydantic import BaseModel, Field
from llm_function_tools import llm_tool


class GetWeatherInput(BaseModel):
    city: str = Field(..., description='City name.')


class GetWeatherOutput(BaseModel):
    forecast: str = Field(..., description='Weather forecast for the city.')


@llm_tool(tags=['weather'])
def get_weather(inputs: GetWeatherInput) -> GetWeatherOutput:
    '''Get current weather for a city.'''
    return GetWeatherOutput(forecast=f'Sunny in {inputs.city}')
""".strip()
)

file_tool_sources = [PythonFileToolSource(str(tool_file))]
file_tool_sources


[PythonFileToolSource(file_path='/tmp/tmpg4e3db4r/weather_tools.py', include_plain_typed=False, location_type='local', package_name=None, package_version=None, module_name=None, loggerLvl=20, logger_name=None, logger_format='%(levelname)s:%(name)s:%(message)s')]

## Create reusable config

Bundle runtime settings and tool sources once, then reuse that config across multiple decorated functions.


In [4]:
runtime_config = LlmRuntimeConfig(
    llm_handler_params={
        "llm_h_type": "ollama",
        "llm_h_params": {
            "connection_string": "http://localhost:11434",
            "model_name": "gpt-oss:20b",
        },
    },
    storage_path="/tmp",
)

llm_config = LlmFunctionConfig(
    runtime=runtime_config,
    tool_sources=tool_sources,
)

llm_config


LlmFunctionConfig(runtime=LlmRuntimeConfig(llm_handler_params={'llm_h_type': 'ollama', 'llm_h_params': {'connection_string': 'http://localhost:11434', 'model_name': 'gpt-oss:20b'}}, storage_path='/tmp', force_replan=False, max_retry=None, reset_loops=None, compare_params=None, test_params=None), tool_sources=[InMemoryToolSource(tools=[<function get_weather at 0x762e88216f80>, <function send_report_email at 0x762e8823c550>, <function query_database at 0x762e8823c820>, <function query_web at 0x762e8823caf0>], location_type='local', package_name=None, package_version=None, origin_ref=None, loggerLvl=20, logger_name=None, logger_format='%(levelname)s:%(name)s:%(message)s')], tool_registry=None)

## Create reusable runtime

Use `LlmFunctionRuntime` when multiple decorated functions or repeated calls should reuse the same initialized `WorkflowAutoAssembler` and in-memory workflow cache.


In [5]:
llm_runtime = LlmFunctionRuntime(config=llm_config)

llm_runtime


LlmFunctionRuntime(available_functions=None, available_callables=None, tool_registry=None, tool_sources=[InMemoryToolSource(tools=[<function get_weather at 0x762e88216f80>, <function send_report_email at 0x762e8823c550>, <function query_database at 0x762e8823c820>, <function query_web at 0x762e8823caf0>], location_type='local', package_name=None, package_version=None, origin_ref=None, loggerLvl=20, logger_name=None, logger_format='%(levelname)s:%(name)s:%(message)s')], config=LlmFunctionConfig(runtime=LlmRuntimeConfig(llm_handler_params={'llm_h_type': 'ollama', 'llm_h_params': {'connection_string': 'http://localhost:11434', 'model_name': 'gpt-oss:20b'}}, storage_path='/tmp', force_replan=False, max_retry=None, reset_loops=None, compare_params=None, test_params=None), tool_sources=[InMemoryToolSource(tools=[<function get_weather at 0x762e88216f80>, <function send_report_email at 0x762e8823c550>, <function query_database at 0x762e8823c820>, <function query_web at 0x762e8823caf0>], locati

## Define workflow input and output schemas

The decorated function body is unused. Its signature and docstring define the target typed function contract.


In [6]:
class WfInputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class WfOutputs(BaseModel):
    city: str = Field(..., description="Name of the city for which the report email was sent.")
    email_sent: bool = Field(..., description="Confirmation that the report email was sent.")
    info : str  = Field(..., description="Information found in the database.")
    message: str = Field(..., description="Optional comments from the email sending process.")

test_params = [
    {
        "inputs": WfInputs(city = "London"),
        "outputs": WfOutputs(
            city = "London",
            info = "Content extracted from the database for Birds in London is ...",
            email_sent = True,
            message = "Email sent to London!"
        )
    },
    {
        "inputs": WfInputs(city = "Sydney"),
        "outputs": WfOutputs(
            city = "Sydney",
            info = "Content extracted from the database for Birds in Sydney is ...",
            email_sent = False,
            message = "Email was not sent to Sydney!"
        )
    }
] 

@llm_function(runtime=llm_runtime, test_params = test_params)
def query_db_and_send_email(input: WfInputs) -> WfOutputs:
    """
    Query  database to find information on birds and get latest weather for the city, then send an email there.
    """
    pass


## Call the generated typed function

On each call, the decorator reuses the shared `LlmFunctionRuntime`, calls `actualize_workflow(...)`, and returns the typed output.


In [7]:
result = query_db_and_send_email(WfInputs(city="London"))
result

WfOutputs(city='London', email_sent=True, info='Content extracted from the database for Birds in London is ...', message='Email sent to London!')

In [8]:
result = query_db_and_send_email(WfInputs(city="Wrocław"))
result


WfOutputs(city='Wrocław', email_sent=False, info='Content extracted from the database for Birds in Wrocław is ...', message='Email was not sent to Wrocław!')

If planning is not possible or failed along the way, a function suppose to return `LlmFunctionError` error.

In [9]:
class WfInputs(BaseModel):
    city: str = Field(..., description="Name of the city.")


class WfOutputs(BaseModel):
    city: str = Field(..., description="Name of the city.")
    summary: str = Field(..., description="Some summary that is not about forcasting.")


@llm_function(config=llm_config)
def impossible_get_city_weather(input: WfInputs) -> WfOutputs:
    """
    Get weather for the provided city and prepare a short user-facing non forcast summary.
    """
    pass

result = impossible_get_city_weather(WfInputs(city="Wrocław"))
result


LlmFunctionError: The only provided function that obtains weather data is get_weather, which returns a forecast string. There is no function available that can transform this forecast into a short user-facing non‑forecast summary as required by the output schema. Therefore the task cannot be completed with the given tools.